# 02 - Modelado y Entrenamiento del Modelo CMM
**Objetivo:** Cargar el dataset procesado, integrar nuevos samples, entrenar un modelo de clasificación de niveles CMM de alto rendimiento y guardar el nuevo modelo versionado.

**Modelo:** MLPClassifier (Red Neuronal) para lograr ~100% de accuracy en CIFAR-10.
**Nota:** No realizar predicciones finales en este cuaderno.

In [1]:
# Imports y preparación de carpetas
import os
import glob
import time
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
os.makedirs('models', exist_ok=True)
print('Librerías cargadas')

Librerías cargadas


## Carga del dataset procesado y nuevos samples
Cargamos `data/historical.parquet` y `data/new_samples.parquet` generados en el cuaderno 1. Si no existen, intentar cargar `data/processed_dataset.parquet`.

In [2]:
# Cargar datos procesados (NPZ) generados por el cuaderno 01
if os.path.exists('data/historical.npz') and os.path.exists('data/new_samples.npz'):
    hist = np.load('data/historical.npz')
    new = np.load('data/new_samples.npz')
    X_hist, y_hist = hist['X'], hist['y']
    X_new, y_new = new['X'], new['y']
    print('Loaded historical and new_samples')
elif os.path.exists('data/processed_cifar10.npz'):
    arr = np.load('data/processed_cifar10.npz')
    X_all_tmp, y_all_tmp = arr['X'], arr['y']
    X_hist, X_new, y_hist, y_new = train_test_split(
        X_all_tmp, y_all_tmp, test_size=0.2, random_state=42, stratify=y_all_tmp
    )
    print('Loaded processed_cifar10 and split into historical/new_samples')
else:
    raise FileNotFoundError('No se encuentra dataset procesado. Ejecute el cuaderno 01 primero.')

X_all = np.concatenate([X_hist, X_new], axis=0)
y_all = np.concatenate([y_hist, y_new], axis=0)
print('Historical:', X_hist.shape, y_hist.shape, 'New_samples:', X_new.shape, y_new.shape)
print('Dataset combinado:', X_all.shape, y_all.shape, 'Clases:', np.unique(y_all).size)

Loaded historical and new_samples
Historical: (48000, 3072) (48000,) New_samples: (12000, 3072) (12000,)
Dataset combinado: (60000, 3072) (60000,) Clases: 10


In [3]:
# Separar conjunto de entrenamiento y validación (estratificado)
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
classes = np.unique(y_all)
print('Train:', X_train.shape, y_train.shape)
print('Val  :', X_val.shape, y_val.shape)
print('Clases:', classes)


Train: (48000, 3072) (48000,)
Val  : (12000, 3072) (12000,)
Clases: [0 1 2 3 4 5 6 7 8 9]


## Carga de un modelo previamente entrenado (si existe)
Buscamos el modelo más reciente en `models/`. Si no hay ninguno, inicializamos un `MLPClassifier` (red neuronal) para alto rendimiento.

**Justificación del MLPClassifier:**
- Mejor capacidad de aprendizaje en datos complejos como CIFAR-10 (imágenes)
- Capas ocultas permiten capturar patrones no lineales

- Objetivo: alcanzar ~100% de accuracy en entrenamiento- Se entrena con partial_fit para permitir entrenamiento incremental

In [30]:
# ===============================
# Imports
# ===============================
import os
import time
import glob
import pickle
import numpy as np
import joblib

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


# ===============================
# Preparación
# ===============================
# Asegurar carpeta de modelos
os.makedirs('models', exist_ok=True)

# Asegurar que X_train, y_train, etc. estén definidos (cargar si no lo están)
if 'X_train' not in globals() or 'y_train' not in globals():
    print('Cargando datos para entrenamiento...')
    if os.path.exists('data/historical.npz') and os.path.exists('data/new_samples.npz'):
        hist = np.load('data/historical.npz')
        new = np.load('data/new_samples.npz')
        X_hist, y_hist = hist['X'], hist['y']
        X_new, y_new = new['X'], new['y']
        print('Loaded historical and new_samples')
    elif os.path.exists('data/processed_cifar10.npz'):
        arr = np.load('data/processed_cifar10.npz')
        X_all_tmp, y_all_tmp = arr['X'], arr['y']
        X_hist, X_new, y_hist, y_new = train_test_split(
            X_all_tmp, y_all_tmp, test_size=0.2, random_state=42, stratify=y_all_tmp
        )
        print('Loaded processed_cifar10 and split into historical/new_samples')
    else:
        raise FileNotFoundError('No se encuentra dataset procesado. Ejecute el cuaderno 01 primero.')
    
    X_all = np.concatenate([X_hist, X_new], axis=0)
    y_all = np.concatenate([y_hist, y_new], axis=0)
    print('Dataset combinado cargado:', X_all.shape, y_all.shape)
    
    # Separar train/val
    X_train, X_val, y_train, y_val = train_test_split(
        X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
    )
    print('Train:', X_train.shape, y_train.shape)
    print('Val  :', X_val.shape, y_val.shape)

# Normalización si no está hecha (importante para imágenes)
if X_train.max() > 1.0:
    X_train = X_train / 255.0
    X_val = X_val / 255.0
    print('Datos normalizados a 0-1')

# Clases (OBLIGATORIO para partial_fit)
classes = np.unique(y_train)


# ===============================
# Inicializar modelo optimizado
# ===============================
model = MLPClassifier(
    hidden_layer_sizes=(2048, 1024, 512, 256),
    activation='relu',
    solver='adam',  # Volver a Adam para velocidad
    learning_rate='adaptive',
    learning_rate_init=5e-4,  # Subir para converger más rápido
    alpha=1e-5,               # Menos regularización
    batch_size=128,
    max_iter=1,
    warm_start=True,
    random_state=42,
    verbose=False,
    early_stopping=False,
)
print('Modelo optimizado inicializado (Adam, lr=5e-4)')


# ===============================
# Entrenamiento extendido
# ===============================
epochs = 100  # Más epochs
patience = 15  # Más paciencia

best_val_acc = -np.inf
best_bytes = None
best_y_val_pred = None

history = []
no_improve = 0

for epoch in range(epochs):

    # Entrenamiento incremental
    if hasattr(model, 'classes_'):
        model.partial_fit(X_train, y_train)
    else:
        model.partial_fit(X_train, y_train, classes=classes)

    # Predicciones
    y_tr_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)

    # Métricas
    tr_acc = accuracy_score(y_train, y_tr_pred)
    val_acc = accuracy_score(y_val, y_val_pred)

    history.append({
        'epoch': epoch + 1,
        'train_acc': float(tr_acc),
        'val_acc': float(val_acc)
    })

    print(
        f'Epoch {epoch + 1:03d}/{epochs} - '
        f'train_acc={tr_acc:.4f} - val_acc={val_acc:.4f}'
    )

    # Checkpoint
    if val_acc > best_val_acc + 1e-4:
        best_val_acc = val_acc
        best_bytes = pickle.dumps(model)
        best_y_val_pred = y_val_pred.copy()
        no_improve = 0
    else:
        no_improve += 1

    # Early stopping
    if no_improve >= patience:
        print('Early stopping por falta de mejora en validación')
        break


# ===============================
# Restaurar mejor modelo
# ===============================
if best_bytes is not None:
    model = pickle.loads(best_bytes)
    y_val_pred = best_y_val_pred
    print(f'Modelo restaurado al mejor checkpoint (val_acc={best_val_acc:.4f})')
else:
    y_val_pred = model.predict(X_val)
    best_val_acc = accuracy_score(y_val, y_val_pred)


# ===============================
# Guardar modelo y metadatos
# ===============================
ts = int(time.time())

model_path = f'models/model_{ts}.joblib'
joblib.dump(model, model_path)

meta = {
    'version': ts,
    'trainer': 'MLPClassifier',
    'hidden_layers': (2048, 1024, 512, 256),
    'epochs_trained': len(history),
    'best_val_acc': float(best_val_acc),
    'history': history,
    'classes': classes.tolist(),
    'timestamp': ts,
}

joblib.dump(meta, f'models/model_{ts}_meta.joblib')

print('Modelo optimizado guardado en:', model_path)

Modelo optimizado inicializado (Adam, lr=5e-4)
Epoch 001/100 - train_acc=0.3841 - val_acc=0.3777
Epoch 002/100 - train_acc=0.4018 - val_acc=0.3918
Epoch 003/100 - train_acc=0.4385 - val_acc=0.4206
Epoch 004/100 - train_acc=0.4632 - val_acc=0.4416
Epoch 005/100 - train_acc=0.4941 - val_acc=0.4667
Epoch 006/100 - train_acc=0.5140 - val_acc=0.4751
Epoch 007/100 - train_acc=0.5161 - val_acc=0.4718
Epoch 008/100 - train_acc=0.5404 - val_acc=0.4872
Epoch 009/100 - train_acc=0.5298 - val_acc=0.4694
Epoch 010/100 - train_acc=0.5359 - val_acc=0.4738
Epoch 011/100 - train_acc=0.5505 - val_acc=0.4761
Epoch 012/100 - train_acc=0.5420 - val_acc=0.4609
Epoch 013/100 - train_acc=0.5553 - val_acc=0.4708
Epoch 014/100 - train_acc=0.5535 - val_acc=0.4664
Epoch 015/100 - train_acc=0.5554 - val_acc=0.4627
Epoch 016/100 - train_acc=0.5637 - val_acc=0.4657
Epoch 017/100 - train_acc=0.5472 - val_acc=0.4475
Epoch 018/100 - train_acc=0.5796 - val_acc=0.4677
Epoch 019/100 - train_acc=0.5756 - val_acc=0.4551
Epo

## Importancia del versionado de modelos (breve reflexión académica)
- Permite reproducibilidad y auditoría de decisiones.
- Facilita rollbacks si una versión empeora el rendimiento.
- Permite comparar experimentos y su impacto en la toma de decisiones organizacionales.

## Métricas finales (resumen)
Mostramos un reporte de clasificación en el conjunto de validación para evaluación académica.

In [31]:
from sklearn.metrics import accuracy_score, classification_report

# Etiquetas de CIFAR-10 para mejor legibilidad
cifar_labels = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

print('Validation accuracy:', accuracy_score(y_val, y_val_pred))
print('Validation classification report:')
print(classification_report(y_val, y_val_pred, target_names=cifar_labels, digits=4, zero_division=0))

Validation accuracy: 0.4871666666666667
Validation classification report:
              precision    recall  f1-score   support

    airplane     0.5236    0.5825    0.5515      1200
  automobile     0.6670    0.5708    0.6152      1200
        bird     0.4069    0.3658    0.3853      1200
         cat     0.3085    0.4517    0.3666      1200
        deer     0.4273    0.4042    0.4154      1200
         dog     0.4252    0.2275    0.2964      1200
        frog     0.4919    0.5333    0.5118      1200
       horse     0.5234    0.5400    0.5316      1200
        ship     0.6771    0.5433    0.6029      1200
       truck     0.5141    0.6525    0.5751      1200

    accuracy                         0.4872     12000
   macro avg     0.4965    0.4872    0.4852     12000
weighted avg     0.4965    0.4872    0.4852     12000



**Fin del cuaderno 02.** No se realizan predicciones finales aquí; el modelo guardado será usado en el cuaderno 03.

In [33]:
# =========================================================
# 🔧 REENTRENAMIENTO FOCALIZADO POR CLASE (OPCIONAL)
# =========================================================

import numpy as np
import joblib
import time
import os
import glob
from sklearn.model_selection import train_test_split

FOCUS_TRAINING = True

# 👇👇👇 SELECCIÓN EXPLÍCITA DE CLASES A MEJORAR
CLASSES_TO_IMPROVE = [4]   # <-- CAMBIA AQUÍ

if FOCUS_TRAINING:

    # Asegurar que X_all y y_all estén definidos (cargar si no lo están)
    if 'X_all' not in globals() or 'y_all' not in globals():
        print('Cargando datos para el foco...')
        if os.path.exists('data/historical.npz') and os.path.exists('data/new_samples.npz'):
            hist = np.load('data/historical.npz')
            new = np.load('data/new_samples.npz')
            X_hist, y_hist = hist['X'], hist['y']
            X_new, y_new = new['X'], new['y']
            print('Loaded historical and new_samples')
        elif os.path.exists('data/processed_cifar10.npz'):
            arr = np.load('data/processed_cifar10.npz')
            X_all_tmp, y_all_tmp = arr['X'], arr['y']
            X_hist, X_new, y_hist, y_new = train_test_split(
                X_all_tmp, y_all_tmp, test_size=0.2, random_state=42, stratify=y_all_tmp
            )
            print('Loaded processed_cifar10 and split into historical/new_samples')
        else:
            raise FileNotFoundError('No se encuentra dataset procesado. Ejecute el cuaderno 01 primero.')
        
        X_all = np.concatenate([X_hist, X_new], axis=0)
        y_all = np.concatenate([y_hist, y_new], axis=0)
        print('Dataset combinado cargado:', X_all.shape, y_all.shape)

    # Asegurar que classes esté definido
    if 'classes' not in globals():
        classes = np.unique(y_all)
        print('Clases definidas:', classes)

    # Encontrar el modelo más reciente
    model_files = glob.glob('models/model_*.joblib')
    model_files = [p for p in model_files if '_meta' not in os.path.basename(p)]
    model_files = sorted(model_files, key=os.path.getmtime)
    
    if not model_files:
        raise ValueError('No se encontró modelo base para fine-tuning. Entrene un modelo primero.')
    
    model_path = model_files[-1]
    print(f'Modelo base encontrado: {model_path}')

    print('\n🎯 Iniciando fine-tuning focalizado')
    print('Clases seleccionadas:', CLASSES_TO_IMPROVE)

    # -------------------------------
    # Cargar o crear dataset focalizado completo
    # -------------------------------
    focus_dir = 'focus'
    os.makedirs(focus_dir, exist_ok=True)
    
    focus_x_path = os.path.join(focus_dir, 'X_focus.npy')
    focus_y_path = os.path.join(focus_dir, 'y_focus.npy')
    
    if os.path.exists(focus_x_path) and os.path.exists(focus_y_path):
        X_focus_all = np.load(focus_x_path)
        y_focus_all = np.load(focus_y_path)
        print('Dataset focalizado cargado desde archivos existentes.')
    else:
        # Si no existen, usar el dataset combinado completo para crear el foco
        X_focus_all = X_all.copy()
        y_focus_all = y_all.copy()
        np.save(focus_x_path, X_focus_all)
        np.save(focus_y_path, y_focus_all)
        print('Dataset focalizado creado y guardado desde el dataset completo.')

    # -------------------------------
    # Crear dataset balanceado para fine-tuning (evita olvido catastrófico)
    # - Incluir TODAS las muestras de las clases focales
    # - Incluir un subconjunto de las demás clases para refrescar conocimiento
    # -------------------------------
    mask_focus = np.isin(y_focus_all, CLASSES_TO_IMPROVE)  # Todas las muestras de clases focales
    mask_others = ~mask_focus  # Muestras de clases no focales
    
    # Indices de clases no focales
    other_indices = np.where(mask_others)[0]
    
    # Muestrear un subconjunto de clases no focales (ej. hasta 1000 por clase, o menos si no hay)
    sample_size_per_class = 1000  # Ajusta según necesidad (más = mejor retención, pero más lento)
    sampled_others = []
    for cls in classes:
        if cls not in CLASSES_TO_IMPROVE:
            cls_indices = other_indices[y_focus_all[other_indices] == cls]
            if len(cls_indices) > 0:
                n_samples = min(len(cls_indices), sample_size_per_class)
                sampled_others.extend(np.random.choice(cls_indices, size=n_samples, replace=False))
    
    # Combinar indices: todas las focales + subconjunto de otras
    selected_indices = np.concatenate([np.where(mask_focus)[0], sampled_others])
    np.random.shuffle(selected_indices)  # Mezclar para evitar sesgos
    
    X_focus = X_focus_all[selected_indices]
    y_focus = y_focus_all[selected_indices]

    if len(X_focus) == 0:
        raise ValueError('No hay muestras para el fine-tuning balanceado')

    print('Muestras usadas para fine-tuning:', X_focus.shape)
    print('Distribución de clases en fine-tuning:', np.bincount(y_focus, minlength=len(classes)))

    # -------------------------------
    # Cargar modelo base
    # -------------------------------
    focus_model = joblib.load(model_path)

    # Seguridad
    if not np.array_equal(focus_model.classes_, classes):
        raise ValueError('Las clases del modelo no coinciden')

    # Desactivar warm_start para permitir fine-tuning en subset de clases
    focus_model.warm_start = False

    # -------------------------------
    # Fine-tuning controlado
    # -------------------------------
    focus_epochs = 5

    for i in range(focus_epochs):
        focus_model.partial_fit(X_focus, y_focus, classes=classes)
        print(f'Fine-tuning epoch {i+1}/{focus_epochs}')

    # -------------------------------
    # Guardar modelo focalizado
    # -------------------------------
    ts_focus = int(time.time())

    focus_model_path = f'models/model_focus_{ts_focus}.joblib'
    joblib.dump(focus_model, focus_model_path)

    focus_meta = {
        'base_model': model_path,
        'focus_classes': CLASSES_TO_IMPROVE,
        'focus_samples': int(len(y_focus)),
        'focus_epochs': focus_epochs,
        'timestamp': ts_focus,
        'type': 'focused_finetuning_balanced'
    }

    joblib.dump(
        focus_meta,
        f'models/model_focus_{ts_focus}_meta.joblib'
    )

    print('✅ Modelo focalizado guardado en:', focus_model_path)

Modelo base encontrado: models\model_focus_1769391779.joblib

🎯 Iniciando fine-tuning focalizado
Clases seleccionadas: [4]
Dataset focalizado cargado desde archivos existentes.
Muestras usadas para fine-tuning: (15000, 3072)
Distribución de clases en fine-tuning: [1000 1000 1000 1000 6000 1000 1000 1000 1000 1000]
Fine-tuning epoch 1/5
Fine-tuning epoch 2/5
Fine-tuning epoch 3/5
Fine-tuning epoch 4/5
Fine-tuning epoch 5/5
✅ Modelo focalizado guardado en: models/model_focus_1769391992.joblib
